In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
## Import Data
sensor_data = pd.read_csv('./data/october_RAW.csv', parse_dates = ['Local Date/Time', 'UTC Date/Time'])
cal_data = pd.read_csv('./data/lab_reference-CO2.csv', parse_dates = ['datetime_utc'])

In [ ]:
## Choose relevant columns
sens_c = sensor_data[['Local Date/Time', 'CO2 (ppm) raw']]
cal_c = cal_data[['datetime_utc', 'ref_co2_ppm']]

In [ ]:
## Choose correct time frame for sensor data (oct 9-12?)
sens_c = sens_c.loc[(
        sens_c['Local Date/Time'] >= pd.Timestamp(2025, 10, 9)
    ) & (
        sens_c['Local Date/Time'] <= pd.Timestamp(2025, 10, 14)
    ) & (
        sens_c['CO2 (ppm) raw'] <= 320
    )
]

In [ ]:
## Find maximum and minimum values for each
display(sens_c.nlargest(n = 10, columns = 'CO2 (ppm) raw'))
display(cal_c.nlargest(n = 10, columns = 'ref_co2_ppm'))

From the eyeball test, we see that the sensor data has a maximum somewhere around 8:30 on Oct 12th. The calibration data seems to show an absolute maximum aronud 6:30 on the same day, although it also has near (and large) values around 2:00 and 4:00.

From this, we can deduce that we need to shift the data backwards by around 120 minutes for our sensor.

In [ ]:
TIME_SHIFT = - pd.Timedelta(hours = 2)

sens_c['Local Date/Time'] = sens_c['Local Date/Time'] + TIME_SHIFT

In [ ]:
## Plot data
fig, ax = plt.subplots()

sens_c.plot(x = 'Local Date/Time', y = 'CO2 (ppm) raw', ax = ax)
cal_c.plot(x = 'datetime_utc', y = 'ref_co2_ppm', ax = ax)

In [ ]:
## Regularising the time series
merged_c = sens_c.set_index('Local Date/Time').reindex(cal_c['datetime_utc'], tolerance=pd.Timedelta(seconds = 30), method = 'nearest')

In [ ]:
merged_c

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit()